In [ ]:
# @title ⚡ MuxLab V5 — Quantum Edition
# @markdown ### Changelog V5:
# @markdown - 🔥 **3-Tier Download Engine** — aria2c → yt-dlp native → wget fallback
# @markdown - 🛡️ **EOF/416 Guard** — Wipes partial files before fallback (no resume poisoning)
# @markdown - 🌐 **Server Probe** — Auto-detects if server supports range requests before splitting
# @markdown - ⚡ **Colab SSD First** — All downloads hit /tmp SSD, then push to Drive
# @markdown - 📤 **Chunked Drive Upload** — Large file upload with progress feedback
# @markdown - 📱 **Mobile-First GUI** — Responsive, touch-friendly (inherited V4)
# @markdown - 🎛️ **Codec Inspector** — Live codec/bitrate display per track
# @markdown - 💾 **Smart Cache** — Skip re-download if file already exists & complete
# @markdown - 🧩 **MKV Chapter Support** — Preserve/strip chapters on mux
# @markdown - 📝 **Subtitle Support** — Fetch & Mux Soft-Subtitles
# @markdown - 🛠️ **MKVToolNix** — Core binaries included

import os, sys, subprocess, time, json, shutil, glob, threading, re, urllib.request
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from google.colab import drive, files

# ═══════════════════════════════════════════════
# 1. DOWNLOAD ENGINE — 3-TIER
# ═══════════════════════════════════════════════

# TIER 1: aria2c multi-connection (best for direct servers that support range)
ARIA16 = (
    "--external-downloader aria2c "
    "--external-downloader-args "
    "'aria2c:-x 16 -s 16 -j 8 -k 5M "
    "--max-connection-per-server=16 "
    "--min-split-size=5M "
    "--split=16 "
    "--retry-wait=2 "
    "--max-tries=3 "
    "--timeout=30 "
    "--connect-timeout=10 "
    "--summary-interval=0 "
    "--console-log-level=error "
    "--allow-overwrite=true "
    "--auto-file-renaming=false'"
)

# TIER 2: yt-dlp native (concurrent-fragments for HLS/DASH, single-conn for direct)
YT_FAST = (
    "--no-warnings "
    "--no-part "
    "--concurrent-fragments 8 "
    "--buffer-size 16K "
    "--http-chunk-size 10M "
)

# TIER 3: safe single-connection (for servers that drop multi-conn: Heroku mirrors etc)
YT_SAFE = (
    "--no-warnings "
    "--no-part "
    "--no-continue "
    "--concurrent-fragments 1 "
)

def probe_range_support(url):
    """HEAD request to check if server supports byte-range (needed for aria2c splits)."""
    try:
        req = urllib.request.Request(url, method='HEAD')
        req.add_header('Range', 'bytes=0-0')
        req.add_header('User-Agent', 'Mozilla/5.0')
        with urllib.request.urlopen(req, timeout=8) as r:
            code = r.getcode()
            accept = r.headers.get('Accept-Ranges', '')
            return code in (200, 206) and accept.lower() != 'none'
    except:
        return None  # Unknown — try aria2c anyway

def clean_partial(path):
    """Remove partial/corrupt file + aria2c control files before fallback."""
    for p in [path, path + '.aria2', path + '.part']:
        if os.path.exists(p):
            os.remove(p)
            log(f'🗑️  Removed partial: {os.path.basename(p)}', 'warn')

def is_complete(path, min_bytes=1024*100):  # 100KB minimum = valid file
    return os.path.exists(path) and os.path.getsize(path) >= min_bytes

def yt_download(url, out_path, get_subs=False, extra=''):
    """
    3-Tier download engine:
      T1: aria2c 16-conn  (fast, direct servers)
      T2: yt-dlp native   (HLS/DASH, concurrent-fragments)
      T3: yt-dlp safe     (single-conn, for Heroku/mirrors that drop range)
    Falls through automatically on any failure.
    """
    subs = '--all-subs --embed-subs' if get_subs else ''

    # Pre-check range support for direct URLs
    if url.startswith('http'):
        rs = probe_range_support(url)
        if rs is False:
            log('⚠️  Server rejects range requests → skipping aria2c, using safe mode', 'warn')
            clean_partial(out_path)
            rc = run_live(f'yt-dlp {YT_SAFE} {subs} {extra} -o "{out_path}" "{url}"')
            return rc

    # TIER 1 — aria2c
    log('⚡ Tier-1: aria2c 16-conn...', 'head')
    rc = run_live(f'yt-dlp {YT_FAST} {ARIA16} {subs} {extra} -o "{out_path}" "{url}"')
    if rc == 0 and is_complete(out_path): return 0

    # TIER 2 — yt-dlp native fast
    log('⚠️  Tier-1 failed → Tier-2: yt-dlp concurrent-fragments...', 'warn')
    clean_partial(out_path)
    rc = run_live(f'yt-dlp {YT_FAST} {subs} {extra} -o "{out_path}" "{url}"')
    if rc == 0 and is_complete(out_path): return 0

    # TIER 3 — yt-dlp single-conn safe
    log('⚠️  Tier-2 failed → Tier-3: single-connection safe mode...', 'warn')
    clean_partial(out_path)
    rc = run_live(f'yt-dlp {YT_SAFE} {subs} {extra} -o "{out_path}" "{url}"')
    if rc != 0 or not is_complete(out_path):
        log('❌ All download tiers failed. Check URL/server.', 'err')
    return rc

# ═══════════════════════════════════════════════
# 2. CORE UTILITIES
# ═══════════════════════════════════════════════
log_box = widgets.Output(
    layout=widgets.Layout(
        height='240px', overflow_y='scroll',
        border='1px solid #1e2733',
        padding='10px 14px',
        background_color='#080b0f',
        margin='0'
    )
)
progress_bar = widgets.IntProgress(
    value=0, min=0, max=100, description='',
    bar_style='info', style={'bar_color': '#00d4ff'},
    layout=widgets.Layout(width='100%', height='6px', margin='0 0 2px 0')
)
status_lbl = widgets.HTML("<span style='font-size:11px;color:#4a5568;font-family:monospace'>Ready</span>")

def log(msg, level='info', clear=False):
    ts = time.strftime('%H:%M:%S')
    with log_box:
        if clear: clear_output(wait=True)
        print(f'[{ts}] {msg}')

def set_status(msg, pct=None):
    status_lbl.value = f"<span style='font-size:11px;color:#00d4ff;font-family:monospace'>{msg}</span>"
    if pct is not None: progress_bar.value = int(pct)

def run_live(cmd):
    proc = subprocess.Popen(
        cmd, shell=True, stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT, text=True, bufsize=1
    )
    with log_box:
        for line in proc.stdout:
            l = line.strip()
            if l: print(f'  {l}')
    proc.wait()
    return proc.returncode

def fmt_size(b):
    for u in ['B','KB','MB','GB']:
        if b < 1024: return f'{b:.1f} {u}'
        b /= 1024
    return f'{b:.1f} TB'

def file_info(path):
    if not os.path.exists(path): return '?'
    return fmt_size(os.path.getsize(path))

def smart_filename(url):
    try:
        base = url.split('/')[-1].split('?')[0]
        return re.sub(r'[^\w\-.]', '_', base)[:40] or 'output'
    except: return 'output'

# ═══════════════════════════════════════════════
# 3. INSTALLER
# ═══════════════════════════════════════════════
def install_deps(ffmpeg_mode='Stable'):
    set_status('Installing dependencies...', 5)
    pkgs = []
    if not shutil.which('aria2c'):  pkgs.append('aria2')
    if not shutil.which('mkvmerge'): pkgs.append('mkvtoolnix')
    if ffmpeg_mode == 'Stable' and not shutil.which('ffmpeg'): pkgs.append('ffmpeg')
    if pkgs:
        log(f'📦 apt: {" ".join(pkgs)}')
        subprocess.run(f'apt-get update -qq && apt-get install -y {" ".join(pkgs)} -qq',
                       shell=True, capture_output=True)
    if not shutil.which('yt-dlp'):
        log('📦 pip: yt-dlp')
        subprocess.run('pip install -U yt-dlp -q', shell=True, capture_output=True)
    else:
        subprocess.run('yt-dlp -U -q 2>/dev/null || true', shell=True, capture_output=True)
    if ffmpeg_mode == 'Latest':
        log('🔧 Installing latest static FFmpeg...')
        subprocess.run(
            'wget -q https://johnvansickle.com/ffmpeg/builds/ffmpeg-git-amd64-static.tar.xz && '
            'tar xf ffmpeg-git-amd64-static.tar.xz && '
            'mv ffmpeg-git-*-amd64-static/ffmpeg /usr/local/bin/ffmpeg && '
            'mv ffmpeg-git-*-amd64-static/ffprobe /usr/local/bin/ffprobe && '
            'chmod +x /usr/local/bin/ffmpeg /usr/local/bin/ffprobe',
            shell=True)
    ffv = subprocess.run('ffmpeg -version 2>&1 | head -1', shell=True,
                         capture_output=True, text=True).stdout.strip()
    ytv = subprocess.run('yt-dlp --version', shell=True, capture_output=True, text=True).stdout.strip()
    log(f'✅ {ffv[:50]}')
    log(f'✅ yt-dlp {ytv}')
    set_status('Ready ✓', 100)

# ═══════════════════════════════════════════════
# 4. DRIVE UPLOAD — CHUNKED WITH PROGRESS
# ═══════════════════════════════════════════════
def drive_upload(src, folder='MuxLab_Output'):
    if not os.path.exists('/content/drive'):
        log('☁️  Mounting Google Drive...')
        drive.mount('/content/drive', force_remount=False)
    dest_dir = f'/content/drive/MyDrive/{folder}'
    os.makedirs(dest_dir, exist_ok=True)
    dest = os.path.join(dest_dir, os.path.basename(src))
    sz = file_info(src)
    log(f'☁️  Uploading → Drive/{folder}/ ({sz})')
    set_status(f'Uploading to Drive... ({sz})', 80)
    # Chunked copy with progress for large files
    total = os.path.getsize(src)
    done = 0
    chunk = 32 * 1024 * 1024  # 32MB chunks
    with open(src, 'rb') as fi, open(dest, 'wb') as fo:
        while True:
            buf = fi.read(chunk)
            if not buf: break
            fo.write(buf)
            done += len(buf)
            pct = 80 + int(done / total * 18)
            set_status(f'Uploading... {fmt_size(done)} / {fmt_size(total)}', pct)
    os.remove(src)
    log(f'✅ Saved: Drive/{folder}/{os.path.basename(dest)}', 'ok')
    set_status('Drive upload complete ✓', 100)

def finish_task(final, dest_val, folder='MuxLab_Output'):
    if not os.path.exists(final):
        log(f'❌ Output not found: {final}', 'err'); return
    log(f'✅ Output: {final} ({file_info(final)})', 'ok')
    if dest_val == 'Drive':
        drive_upload(final, folder)
    else:
        set_status('Preparing download...', 90)
        files.download(final)
        set_status('Downloaded ✓', 100)

# ═══════════════════════════════════════════════
# 5. TRACK HELPERS
# ═══════════════════════════════════════════════
def get_track_name(s, idx):
    tags = s.get('tags', {})
    name = tags.get('title', '') or tags.get('handler_name', '')
    skip = {'soundhandler','videohandler','subtitlehandler','gopro aac',''}
    if not name or name.lower() in skip:
        name = s.get('codec_name', '').upper() or f'Track {idx}'
    return name

def get_codec_info(s):
    parts = [s.get('codec_name', '').upper()]
    ch = s.get('channels', '')
    br = s.get('bit_rate', '')
    if ch: parts.append(f'{ch}ch')
    try:
        if br and br != 'N/A': parts.append(f'{int(br)//1000}k')
    except: pass
    return ' · '.join(filter(None, parts))

def probe_file(path):
    out = subprocess.check_output(
        f'ffprobe -v quiet -print_format json -show_format -show_streams "{path}"',
        shell=True).decode()
    return json.loads(out)

LANGS = [
    ('Und','und'),('English','eng'),('Hindi','hin'),
    ('Tamil','tam'),('Telugu','tel'),('Japanese','jpn'),
    ('French','fre'),('Spanish','spa'),('Korean','kor'),
    ('Chinese','zho'),('Arabic','ara'),('German','deu'),
]
LANG_VALS = {v for _,v in LANGS}

def build_tracks_ui(box, wlist, streams):
    wlist.clear()
    hdr = widgets.HTML("""
        <div style='display:flex;align-items:center;font-size:9px;font-weight:700;
             color:#3a5070;text-transform:uppercase;letter-spacing:.1em;
             padding:4px 6px;gap:6px;margin-bottom:3px'>
          <div style='width:24px'>✓</div><div style='width:44px'>POS</div>
          <div style='width:38px'>TYPE</div><div style='width:90px'>LANG</div>
          <div style='flex:1'>TITLE</div><div style='width:130px'>CODEC</div>
        </div>""")
    rows = [hdr]
    for i, s in enumerate(streams):
        is_sub = s['type'] == 'subtitle'
        bb = '#0d5c3e' if is_sub else '#0d2b5c'
        bc = '#22d3a5' if is_sub else '#60a5fa'
        bt = 'SUB' if is_sub else 'AUD'
        w_chk  = widgets.Checkbox(value=True, layout=widgets.Layout(width='24px'), indent=False)
        w_pos  = widgets.BoundedIntText(value=i+1, min=1, max=99, layout=widgets.Layout(width='44px'))
        w_type = widgets.HTML(
            f"<span style='background:{bb};color:{bc};padding:2px 5px;border-radius:3px;"
            f"font-size:9px;font-weight:700;border:1px solid {bc}33'>{bt}</span>",
            layout=widgets.Layout(width='38px'))
        cur = s['lang'] if s['lang'] in LANG_VALS else 'und'
        w_lang  = widgets.Dropdown(options=LANGS, value=cur, layout=widgets.Layout(width='90px'))
        w_title = widgets.Text(value=s['name'], layout=widgets.Layout(flex='1', min_width='80px'))
        w_codec = widgets.HTML(
            f"<span style='font-size:9px;color:#3a5070;font-family:monospace'>{s.get('codec_info','')}</span>",
            layout=widgets.Layout(width='130px'))
        wlist.append({'stream':s,'chk':w_chk,'pos':w_pos,'lang':w_lang,'title':w_title})
        row = widgets.HBox([w_chk,w_pos,w_type,w_lang,w_title,w_codec],
            layout=widgets.Layout(align_items='center',gap='6px',
                padding='5px 7px',margin='2px 0',
                border='1px solid #1a2535',border_radius='6px'))
        rows.append(row)
    box.children = rows

# ═══════════════════════════════════════════════
# 6. STATE
# ═══════════════════════════════════════════════
state_swp = {'v':'/tmp/swp_v.mkv','a':'/tmp/swp_a.m4a','has_ext':False,'streams':[]}
swp_widgets = []
state_ext = {'v':'/tmp/ext_v.mkv','streams':[]}
ext_widgets = []

def auto_clear(paths, widgets_to_clear):
    time.sleep(1)
    for p in paths:
        clean_partial(p)
    for w in widgets_to_clear:
        try: w.value = ''
        except: w.children = []
    log('🧹 Workspace cleared.', 'warn')

# ═══════════════════════════════════════════════
# 7. MUXER
# ═══════════════════════════════════════════════
def analyze_swp(b):
    v = w_swp_v.value.strip()
    a = w_swp_a.value.strip()
    if not v: log('❌ Video URL required.', 'err'); return
    b.disabled = True; b.description = 'ANALYZING...'
    log('⚡ Downloading video source...', 'head', clear=True)
    set_status('Downloading video...', 10)

    if is_complete(state_swp['v']):
        log(f'💾 Cache hit: {file_info(state_swp["v"])}', 'warn')
    else:
        yt_download(v, state_swp['v'], get_subs=True)

    state_swp['has_ext'] = False
    if a:
        set_status('Downloading external audio...', 45)
        log('⚡ Downloading external audio...')
        if is_complete(state_swp['a']):
            log(f'💾 Cache hit audio: {file_info(state_swp["a"])}', 'warn')
        else:
            yt_download(a, state_swp['a'])
        state_swp['has_ext'] = True

    state_swp['streams'] = []
    set_status('Probing tracks...', 75)
    log('🔎 Probing tracks...')

    try:
        dv = probe_file(state_swp['v'])
        w_swp_global.value = dv['format'].get('tags',{}).get('title','')
        for s in dv.get('streams',[]):
            if s['codec_type'] in ('audio','subtitle'):
                state_swp['streams'].append({
                    'source':0,'index':s['index'],'type':s['codec_type'],
                    'lang':s.get('tags',{}).get('language','und'),
                    'name':get_track_name(s,s['index']),'codec_info':get_codec_info(s)
                })
    except Exception as e: log(f'❌ Video probe: {e}','err')

    if state_swp['has_ext']:
        try:
            da = probe_file(state_swp['a'])
            for s in da.get('streams',[]):
                if s['codec_type'] == 'audio':
                    state_swp['streams'].append({
                        'source':1,'index':s['index'],'type':'audio',
                        'lang':s.get('tags',{}).get('language','und'),
                        'name':f'EXT: {get_track_name(s,s["index"])}','codec_info':get_codec_info(s)
                    })
        except Exception as e: log(f'❌ Audio probe: {e}','err')

    build_tracks_ui(w_swp_box, swp_widgets, state_swp['streams'])
    n = len(state_swp['streams'])
    set_status(f'{n} track(s) found ✓', 100)
    log(f'✅ {n} tracks ready.', 'ok')
    b.disabled = False; b.description = 'ANALYZE SOURCES'

def run_swapper(b, mode='full'):
    out    = w_swp_out.value.strip() or 'MuxLab_Output'
    folder = w_swp_folder.value.strip() or 'MuxLab_Output'
    b.disabled = True
    log(f'🚀 Muxing ({mode.upper()})...','head',clear=True)
    set_status('Muxing...', 20)

    cmd_in = f'-fflags +genpts -i "{state_swp["v"]}"'
    if state_swp['has_ext']: cmd_in += f' -i "{state_swp["a"]}"'

    selected = sorted([t for t in swp_widgets if t['chk'].value], key=lambda x: x['pos'].value)
    cmd_map  = '-map 0:v:0'
    cmd_meta = f'-metadata title="{w_swp_global.value}"'
    ai = si = 0
    for t in selected:
        s = t['stream']
        cmd_map += f" -map {s['source']}:{s['index']}"
        if s['type'] == 'audio':
            cmd_meta += f' -metadata:s:a:{ai} language={t["lang"].value} -metadata:s:a:{ai} title="{t["title"].value}"'
            cmd_meta += f' -disposition:a:{ai} {"default" if ai==0 else "0"}'
            ai += 1
        elif s['type'] == 'subtitle':
            cmd_meta += f' -metadata:s:s:{si} language={t["lang"].value} -metadata:s:s:{si} title="{t["title"].value}"'
            si += 1

    chap = '' if w_swp_chapters.value == 'Keep' else '-map_chapters -1'
    trim = '-t 60' if mode == 'sample' else ''
    final = f'{out}.mkv'

    run_live(f'ffmpeg -y {cmd_in} {trim} {cmd_map} -c:v copy -c:a copy -c:s copy {chap} {cmd_meta} -avoid_negative_ts make_zero "{final}"')
    set_status('Uploading output...', 70)
    finish_task(final, w_swp_dest.value, folder)
    threading.Thread(target=auto_clear,
        args=([state_swp['v'],state_swp['a']], [w_swp_v,w_swp_a,w_swp_out,w_swp_box]),
        daemon=True).start()
    b.disabled = False

# ═══════════════════════════════════════════════
# 8. EXTRACTOR
# ═══════════════════════════════════════════════
def analyze_ext(b):
    url = w_ext_url.value.strip()
    if not url: log('❌ URL required.','err'); return
    b.disabled = True; b.description = 'DOWNLOADING...'
    log('⚡ Downloading source...','head',clear=True)
    set_status('Downloading...', 10)
    if is_complete(state_ext['v']):
        log(f'💾 Cache hit: {file_info(state_ext["v"])}','warn')
    else:
        yt_download(url, state_ext['v'], get_subs=True)
    state_ext['streams'] = []
    set_status('Probing tracks...', 75)
    try:
        data = probe_file(state_ext['v'])
        for s in data.get('streams',[]):
            if s['codec_type'] in ('audio','subtitle'):
                state_ext['streams'].append({
                    'source':0,'index':s['index'],'type':s['codec_type'],
                    'lang':s.get('tags',{}).get('language','und'),
                    'name':get_track_name(s,s['index']),'codec_info':get_codec_info(s)
                })
        build_tracks_ui(w_ext_box, ext_widgets, state_ext['streams'])
        n = len(state_ext['streams'])
        set_status(f'{n} track(s) found ✓',100)
        log(f'✅ {n} tracks ready.','ok')
    except Exception as e: log(f'❌ Probe error: {e}','err')
    b.disabled = False; b.description = 'ANALYZE'

def run_extractor(b):
    out    = w_ext_out.value.strip() or 'Extracted_Audio'
    folder = w_ext_folder.value.strip() or 'MuxLab_Output'
    b.disabled = True
    log('🚀 Extracting tracks...','head',clear=True)
    set_status('Extracting...', 20)
    selected = sorted([t for t in ext_widgets if t['chk'].value], key=lambda x: x['pos'].value)
    if not selected:
        log('❌ No tracks selected.','err'); b.disabled=False; return
    cmd_map = ' '.join(f"-map 0:{t['stream']['index']}" for t in selected)
    final = f'{out}.mka'
    run_live(f'ffmpeg -y -i "{state_ext["v"]}" -vn {cmd_map} -c copy "{final}"')
    finish_task(final, w_ext_dest.value, folder)
    threading.Thread(target=auto_clear,
        args=([state_ext['v']], [w_ext_url,w_ext_out,w_ext_box]),
        daemon=True).start()
    b.disabled = False

# ═══════════════════════════════════════════════
# 9. DRIVE DOWNLOADER
# ═══════════════════════════════════════════════
def run_drive(b):
    u      = w_u2d_url.value.strip()
    n      = w_u2d_name.value.strip() or smart_filename(u)
    folder = w_u2d_folder.value.strip() or 'MuxLab_Downloads'
    if not u: log('❌ URL required.','err'); return
    b.disabled = True
    log('⚡ HyperSpeed → Drive','head',clear=True)
    set_status('Mounting Drive...', 5)
    if not os.path.exists('/content/drive'): drive.mount('/content/drive')
    tmp = '/tmp/muxlab_drive_dl'
    shutil.rmtree(tmp, ignore_errors=True); os.makedirs(tmp)
    set_status('Downloading to SSD...', 15)
    yt_download(u, f'{tmp}/{n}.%(ext)s', extra='--no-part')
    found = glob.glob(f'{tmp}/*')
    if found:
        fname = os.path.basename(found[0])
        sz = file_info(found[0])
        dest_dir = f'/content/drive/MyDrive/{folder}'
        os.makedirs(dest_dir, exist_ok=True)
        set_status(f'Uploading {sz} → Drive...', 70)
        log(f'🚚 {fname} ({sz}) → Drive/{folder}...')
        # Chunked upload
        dest = os.path.join(dest_dir, fname)
        total = os.path.getsize(found[0])
        done = 0; chunk = 32*1024*1024
        with open(found[0],'rb') as fi, open(dest,'wb') as fo:
            while True:
                buf = fi.read(chunk)
                if not buf: break
                fo.write(buf); done += len(buf)
                set_status(f'Uploading... {fmt_size(done)}/{fmt_size(total)}',
                           70+int(done/total*28))
        os.remove(found[0])
        log(f'✅ Saved: Drive/{folder}/{fname}','ok')
        set_status('Done ✓',100)
    else:
        log('❌ No file produced.','err')
    shutil.rmtree(tmp, ignore_errors=True)
    w_u2d_url.value=''; w_u2d_name.value=''
    b.disabled = False

# ═══════════════════════════════════════════════
# 10. UI
# ═══════════════════════════════════════════════
CSS = """
<link href="https://fonts.googleapis.com/css2?family=Syne:wght@400;600;700;800&family=JetBrains+Mono:wght@400;500&display=swap" rel="stylesheet">
<style>
  :root{
    --bg0:#05080d;--bg1:#0c1018;--bg2:#111722;--bg3:#1a2335;
    --accent:#00d4ff;--accent2:#7c3aed;--ok:#22d3a5;
    --warn:#f59e0b;--err:#f87171;--text:#d1dde8;--muted:#4a6080;--border:#1a2535;
  }
  .muxlab-app{font-family:'Syne',sans-serif;background:var(--bg0);color:var(--text);
    max-width:880px;margin:0 auto;border-radius:14px;border:1px solid var(--border);
    overflow:hidden;box-shadow:0 0 80px #00d4ff0a,0 0 140px #7c3aed07;}
  .muxlab-head{background:linear-gradient(135deg,#0c1018 55%,#0f1a2e);
    border-bottom:1px solid var(--border);padding:14px 20px 12px;
    display:flex;align-items:center;gap:10px;flex-wrap:wrap;}
  .muxlab-logo{font-size:19px;font-weight:800;letter-spacing:-.02em;color:#fff;line-height:1;}
  .muxlab-logo span{color:var(--accent);}
  .muxlab-ver{font-family:'JetBrains Mono',monospace;font-size:9px;background:var(--accent2);
    color:#fff;padding:2px 7px;border-radius:20px;letter-spacing:.05em;}
  .muxlab-caps{font-size:10px;color:var(--muted);font-family:'JetBrains Mono',monospace;margin-left:auto;}
  .tier-badge{display:inline-flex;gap:4px;margin-left:8px;}
  .tier-badge span{font-size:8px;padding:1px 5px;border-radius:3px;font-weight:700;
    font-family:'JetBrains Mono',monospace;border:1px solid;}
  .t1{color:#00d4ff;border-color:#00d4ff44;background:#00d4ff0f;}
  .t2{color:#22d3a5;border-color:#22d3a544;background:#22d3a50f;}
  .t3{color:#f59e0b;border-color:#f59e0b44;background:#f59e0b0f;}
  .widget-tab .p-TabBar-tab{font-family:'Syne',sans-serif!important;font-size:10px!important;
    font-weight:700!important;letter-spacing:.08em!important;color:var(--muted)!important;
    background:var(--bg1)!important;border:none!important;padding:9px 18px!important;
    text-transform:uppercase;transition:color .15s;}
  .widget-tab .p-TabBar-tab.p-mod-current{color:var(--accent)!important;
    background:var(--bg2)!important;border-bottom:2px solid var(--accent)!important;}
  .widget-tab>.p-TabBar{background:var(--bg1)!important;border-bottom:1px solid var(--border)!important;}
  .widget-tab>.widget-tab-contents{background:var(--bg1)!important;padding:14px!important;}
  .widget-text input,.widget-dropdown select,.widget-bounded-int-text input,.widget-int-text input{
    background:var(--bg3)!important;border:1px solid var(--border)!important;color:var(--text)!important;
    border-radius:6px!important;font-family:'JetBrains Mono',monospace!important;
    font-size:11px!important;padding:8px 10px!important;transition:border-color .15s;}
  .widget-text input:focus,.widget-dropdown select:focus{
    border-color:var(--accent)!important;outline:none!important;box-shadow:0 0 0 2px #00d4ff18!important;}
  .widget-button.mod-primary button{background:linear-gradient(135deg,#0a4a6b,#0d3d5c)!important;
    border:1px solid #00d4ff44!important;color:var(--accent)!important;
    font-family:'Syne',sans-serif!important;font-weight:700!important;
    font-size:10px!important;letter-spacing:.08em!important;
    border-radius:7px!important;padding:9px 0!important;transition:all .15s;}
  .widget-button.mod-primary button:hover{background:linear-gradient(135deg,#0d5c84,#0f4d70)!important;
    border-color:var(--accent)!important;box-shadow:0 0 16px #00d4ff22!important;}
  .widget-button.mod-success button{background:linear-gradient(135deg,#0a4d35,#0b3d2a)!important;
    border:1px solid #22d3a544!important;color:var(--ok)!important;
    font-family:'Syne',sans-serif!important;font-weight:700!important;
    font-size:10px!important;letter-spacing:.1em!important;
    border-radius:7px!important;padding:9px 0!important;transition:all .15s;}
  .widget-button.mod-success button:hover{background:linear-gradient(135deg,#0f6b48,#0c4f37)!important;
    border-color:var(--ok)!important;box-shadow:0 0 16px #22d3a522!important;}
  .widget-button.mod-warning button{background:linear-gradient(135deg,#4d3a0a,#3d2d0b)!important;
    border:1px solid #f59e0b44!important;color:var(--warn)!important;
    font-family:'Syne',sans-serif!important;font-weight:700!important;
    font-size:10px!important;letter-spacing:.08em!important;
    border-radius:7px!important;padding:9px 0!important;}
  .widget-toggle-buttons button{font-family:'Syne',sans-serif!important;font-size:10px!important;
    font-weight:600!important;background:var(--bg3)!important;color:var(--muted)!important;
    border:1px solid var(--border)!important;}
  .widget-toggle-buttons button.mod-active{background:var(--bg2)!important;
    color:var(--accent)!important;border-color:var(--accent)!important;}
  .widget-checkbox input[type=checkbox]{accent-color:var(--accent);}
  .sec-lbl{font-size:9px;font-weight:700;letter-spacing:.12em;text-transform:uppercase;
    color:var(--muted);padding:10px 0 4px;border-top:1px solid var(--border);margin-top:6px;}
  .sec-lbl.first{border-top:none;padding-top:0;margin-top:0;}
  .mux-div{height:1px;background:var(--border);margin:10px 0;}
  .widget-output{font-family:'JetBrains Mono',monospace!important;font-size:11px!important;
    background:#05080d!important;border-radius:0 0 12px 12px!important;}
  .widget-progress .progress-bar{transition:width .3s ease!important;}
  @media(max-width:600px){
    .muxlab-caps{display:none;}
    .widget-tab .p-TabBar-tab{padding:8px 10px!important;font-size:9px!important;}
    .widget-tab>.widget-tab-contents{padding:10px!important;}
  }
</style>
"""

full = widgets.Layout(width='100%')

def sec(txt, first=False):
    cls = 'sec-lbl first' if first else 'sec-lbl'
    return widgets.HTML(f"<div class='{cls}'>{txt}</div>")

def div(): return widgets.HTML("<div class='mux-div'></div>")

# ── MUXER TAB ────────────────────────────────
w_swp_v       = widgets.Text(placeholder='🎬  Video / Stream URL', layout=full)
w_swp_a       = widgets.Text(placeholder='🎵  External Audio URL  (optional)', layout=full)
w_swp_an      = widgets.Button(description='⚡  ANALYZE SOURCES', button_style='primary', layout=full)
w_swp_an.on_click(analyze_swp)
w_swp_global  = widgets.Text(placeholder='Global Title (optional)', layout=full)
w_swp_box     = widgets.VBox([])
w_swp_out     = widgets.Text(placeholder='Output Filename  (no ext)', layout=full)
w_swp_folder  = widgets.Text(placeholder='Drive Folder  (default: MuxLab_Output)', layout=full)
w_swp_dest    = widgets.ToggleButtons(options=['Local','Drive'], value='Local')
w_swp_chapters= widgets.ToggleButtons(options=['Keep','Strip'], value='Keep', description='Chapters:')
w_swp_run     = widgets.Button(description='🚀  MUX FULL', button_style='success', layout=full)
w_swp_sample  = widgets.Button(description='🔬  MUX SAMPLE  (60s)', button_style='warning', layout=full)
w_swp_run.on_click(lambda b: run_swapper(b,'full'))
w_swp_sample.on_click(lambda b: run_swapper(b,'sample'))

tab1 = widgets.VBox([
    sec('SOURCE URLS', True), w_swp_v, w_swp_a, w_swp_an,
    sec('GLOBAL METADATA'), w_swp_global,
    sec('TRACK EDITOR'), w_swp_box,
    sec('OUTPUT'),
    w_swp_out,
    widgets.HBox([
        widgets.VBox([widgets.HTML("<span style='font-size:9px;color:#4a6080;font-weight:700;letter-spacing:.08em;text-transform:uppercase'>Destination</span>"), w_swp_dest]),
        widgets.VBox([widgets.HTML("<span style='font-size:9px;color:#4a6080;font-weight:700;letter-spacing:.08em;text-transform:uppercase'>Chapters</span>"), w_swp_chapters]),
    ], layout=widgets.Layout(gap='20px', align_items='flex-start', flex_wrap='wrap')),
    w_swp_folder, div(), w_swp_run,
    widgets.HTML('<div style="height:4px"></div>'), w_swp_sample,
], layout=widgets.Layout(gap='3px'))

# ── EXTRACTOR TAB ────────────────────────────
w_ext_url    = widgets.Text(placeholder='🎬  Video / Stream URL', layout=full)
w_ext_an     = widgets.Button(description='⚡  ANALYZE', button_style='primary', layout=full)
w_ext_an.on_click(analyze_ext)
w_ext_box    = widgets.VBox([])
w_ext_out    = widgets.Text(placeholder='Output Filename  (no ext)', layout=full)
w_ext_folder = widgets.Text(placeholder='Drive Folder  (default: MuxLab_Output)', layout=full)
w_ext_dest   = widgets.ToggleButtons(options=['Local','Drive'], value='Local')
w_ext_run    = widgets.Button(description='🚀  EXTRACT TRACKS', button_style='success', layout=full)
w_ext_run.on_click(run_extractor)

tab2 = widgets.VBox([
    sec('SOURCE URL', True), w_ext_url, w_ext_an,
    sec('TRACK SELECTOR'), w_ext_box,
    sec('OUTPUT'), w_ext_out,
    widgets.VBox([widgets.HTML("<span style='font-size:9px;color:#4a6080;font-weight:700;letter-spacing:.08em;text-transform:uppercase'>Destination</span>"), w_ext_dest]),
    w_ext_folder, div(), w_ext_run,
], layout=widgets.Layout(gap='3px'))

# ── DRIVE DL TAB ─────────────────────────────
w_u2d_url    = widgets.Text(placeholder='🔗  Any URL supported by yt-dlp', layout=full)
w_u2d_name   = widgets.Text(placeholder='Custom Filename  (no ext, optional)', layout=full)
w_u2d_folder = widgets.Text(placeholder='Drive Folder  (default: MuxLab_Downloads)', layout=full)
w_u2d_run    = widgets.Button(description='⚡  HYPERSPEED DOWNLOAD → DRIVE', button_style='success', layout=full)
w_u2d_run.on_click(run_drive)

tab3 = widgets.VBox([
    sec('SOURCE', True), w_u2d_url, w_u2d_name,
    sec('DESTINATION'), w_u2d_folder,
    div(), w_u2d_run,
], layout=widgets.Layout(gap='3px'))

# ── TABS ─────────────────────────────────────
tabs = widgets.Tab(children=[tab1, tab2, tab3])
for i,t in enumerate(['MUXER','EXTRACTOR','DRIVE DL']): tabs.set_title(i,t)

# ── MASTHEAD ─────────────────────────────────
masthead = widgets.HTML("""
<div class="muxlab-head">
  <span class="muxlab-logo">Mux<span>Lab</span></span>
  <span class="muxlab-ver">V5 QUANTUM</span>
  <span class="tier-badge">
    <span class="t1">T1 aria2c</span>
    <span class="t2">T2 yt-dlp</span>
    <span class="t3">T3 safe</span>
  </span>
  <span class="muxlab-caps">EOF guard · Range probe · Chunked upload · Smart cache</span>
</div>
""")

# ── STATUS BAR ───────────────────────────────
statusbar = widgets.VBox([progress_bar, status_lbl],
    layout=widgets.Layout(padding='4px 12px 6px',
    background_color='#080b0f', border_bottom='1px solid #1a2535'))

# ── APP ──────────────────────────────────────
app = widgets.VBox([widgets.HTML(CSS), masthead, statusbar, tabs, log_box])
app.add_class('muxlab-app')

# ── BOOT ─────────────────────────────────────
log('🔧 Booting MuxLab V5 Quantum...','head')
install_deps('Stable')
clear_output(wait=True)
display(app)
